In [2]:
# Step 1 – Install Academic Research Tools
%pip install -q \
  llama-index \
  llama-index-llms-replicate \
  llama-index-embeddings-huggingface \
  llama-index-readers-file \
  llama-index-packs-fusion-retriever \
  sentence-transformers \
  huggingface_hub[hf_xet] \
  hf_xet \
  certifi \
  python-certifi-win32 \
  truststore \
  nest-asyncio \
  requests \
  replicate \
  pytesseract \
  pdf2image \
  Pillow \
  PyMuPDF

import nest_asyncio
nest_asyncio.apply()
print("✅ Installation complete (including OCR deps).")

# NOTE: System dependencies required for OCR:
# - Tesseract OCR executable (installable from https://github.com/tesseract-ocr/tesseract)
# - Poppler utils (for pdf2image) available via package managers or https://poppler.freedesktop.org/
# If those executables are not installed, the OCR fallback will print instructions.

Note: you may need to restart the kernel to use updated packages.
✅ Installation complete (including OCR deps).



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# Console / Logger helper (widget-free for VS Code compatibility)
from datetime import datetime
import logging
import glob
import os
import shutil

def console_log(msg, level='INFO'):
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{ts}] {level}: {msg}")

def resolve_tesseract_cmd() -> str | None:
    """Find and configure a usable Tesseract executable path."""
    candidates = []

    env_cmd = os.getenv("TESSERACT_CMD", "").strip()
    if env_cmd and os.path.exists(env_cmd):
        candidates.append(env_cmd)

    path_cmd = shutil.which("tesseract")
    if path_cmd:
        candidates.append(path_cmd)

    if os.name == "nt":
        default_paths = [
            r"C:\Program Files\Tesseract-OCR\tesseract.exe",
            r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe",
        ]
        for p in default_paths:
            if os.path.exists(p):
                candidates.append(p)

        local_app_data = os.getenv("LOCALAPPDATA", "")
        if local_app_data:
            winget_pattern = os.path.join(
                local_app_data,
                "Microsoft",
                "WinGet",
                "Packages",
                "*",
                "**",
                "tesseract.exe",
            )
            winget_matches = glob.glob(winget_pattern, recursive=True)
            candidates.extend(sorted(winget_matches))

    for cmd in candidates:
        if os.path.exists(cmd):
            os.environ["TESSERACT_CMD"] = cmd
            try:
                import pytesseract
                pytesseract.pytesseract.tesseract_cmd = cmd
            except Exception:
                pass
            return cmd

    return None

class ConsoleHandler(logging.Handler):
    def emit(self, record):
        console_log(self.format(record), record.levelname)

root_logger = logging.getLogger()
if not any(isinstance(h, ConsoleHandler) for h in root_logger.handlers):
    handler = ConsoleHandler()
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s', datefmt='%H:%M:%S')
    handler.setFormatter(formatter)
    root_logger.addHandler(handler)

root_logger.setLevel(logging.INFO)
console_log("Console initialized - logs will appear in cell output.", "OK")

[2026-04-02 12:16:02] OK: Console initialized - logs will appear in cell output.


In [8]:
# Diagnostics: verify Tesseract and Poppler (pdf2image) availability
import shutil
import os

console_log("Running diagnostics: checking system OCR dependencies...", "INFO")

# Tesseract check
tess_path = resolve_tesseract_cmd()
if tess_path:
    try:
        import pytesseract
        v = pytesseract.get_tesseract_version()
        console_log(f"Tesseract found: {tess_path} - version {v}", "OK")
    except Exception as e:
        console_log(f"Tesseract found at {tess_path} but pytesseract error: {e}", "WARN")
else:
    console_log("Tesseract executable not found in PATH or common install locations.", "ERROR")
    console_log(
        "Install Tesseract: https://github.com/tesseract-ocr/tesseract, `winget install UB-Mannheim.TesseractOCR`, or `choco install tesseract`",
        "INFO",
    )

# Poppler check (pdftoppm or pdftocairo)
poppler_bin = shutil.which("pdftoppm") or shutil.which("pdftocairo")
if poppler_bin:
    console_log(f"Poppler utility found: {poppler_bin}", "OK")
else:
    console_log("Poppler utilities (pdftoppm/pdftocairo) not found in PATH.", "ERROR")
    console_log("Install Poppler: https://poppler.freedesktop.org/ or `choco install poppler`", "INFO")

# Optional pdf2image test if a sample PDF exists
sample_pdf = os.path.join("academic_data", "source_material.pdf")
if os.path.exists(sample_pdf):
    if poppler_bin:
        try:
            from pdf2image import convert_from_path
            imgs = convert_from_path(sample_pdf, dpi=50, first_page=1, last_page=1)
            console_log("pdf2image conversion test succeeded (Poppler working).", "OK")
        except Exception as e:
            console_log(f"pdf2image conversion test failed: {e}", "ERROR")
    else:
        console_log("Skipping pdf2image test because Poppler not found.", "WARN")
else:
    console_log(f"No sample PDF at {sample_pdf}; skipping conversion test.", "INFO")

console_log("Diagnostics complete.", "OK")


[2026-04-02 12:16:12] INFO: Running diagnostics: checking system OCR dependencies...
[2026-04-02 12:16:12] ERROR: Tesseract executable not found in PATH or common install locations.
[2026-04-02 12:16:12] INFO: Install Tesseract: https://github.com/tesseract-ocr/tesseract, `winget install UB-Mannheim.TesseractOCR`, or `choco install tesseract`
[2026-04-02 12:16:12] OK: Poppler utility found: C:\Program Files\poppler-25.12.0\Library\bin\pdftoppm.EXE
[2026-04-02 12:16:12] OK: pdf2image conversion test succeeded (Poppler working).
[2026-04-02 12:16:12] OK: Diagnostics complete.


In [19]:
# Step 2: Configure IBM Granite & Security Guardrails
import os
from getpass import getpass
import certifi
from llama_index.core import Settings
from llama_index.llms.replicate import Replicate
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


# Networking hardening for enterprise SSL + HF transport
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
os.environ.setdefault("CURL_CA_BUNDLE", certifi.where())

# On Windows/corporate networks, this merges Windows cert store into certifi trust
try:
    import certifi_win32  # noqa: F401
    print("✅ Windows certificate store bridge enabled (certifi-win32).")
except Exception as e:
    print(f"⚠️ certifi-win32 not active ({e}); using certifi defaults.")

# Enter your REPLICATE_API_KEY safely (env var first, then prompt)
replicate_token = os.getenv("REPLICATE_API_TOKEN", "").strip()
if not replicate_token:
    try:
        replicate_token = getpass("Enter REPLICATE_API_TOKEN: ").strip()
    except Exception:
        replicate_token = input("Enter REPLICATE_API_TOKEN: ").strip()

if not replicate_token:
    raise ValueError("REPLICATE_API_TOKEN is required to continue.")

os.environ["REPLICATE_API_TOKEN"] = replicate_token

# ACADEMIC SHIELD FIX: Granite 3.1 with Extended Patience
llm = Replicate(
    model="ibm-granite/granite-3.1-8b-instruct",
    temperature=0.1, 
    context_window=128000,
    is_chat_model=True,
    request_timeout=600.0, # Increased to 10 minutes for complex Statistics/Leadership PDFs
    system_prompt=(
        "You are an academic research assistant. Answer ONLY using the provided context. "
        "If the answer is not in the text, say 'Information not found in source.' "
        "Provide facts in 2-4 evidence paragraphs with bullets and with absolute objectivity."
    )
)

# Embedding model with robust fallback path
try:
    embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
    print("✅ Primary embedding model loaded: BAAI/bge-small-en-v1.5")
except Exception as e1:
    print(f"⚠️ Primary embedding load failed: {e1}")
    try:
        embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
        print("✅ Fallback embedding model loaded: all-MiniLM-L6-v2")
    except Exception as e2:
        from llama_index.core.embeddings import MockEmbedding
        embed_model = MockEmbedding(embed_dim=384)
        print(f"⚠️ HF downloads unavailable; using MockEmbedding fallback ({e2}).")

Settings.llm = llm
Settings.embed_model = embed_model

print("🚀 Granite 3.1 Ready with Academic Shield (Timeout: 300s)")

✅ Windows certificate store bridge enabled (certifi-win32).
[2026-04-02 16:39:24] INFO: 16:39:24 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
[2026-04-02 16:39:24] INFO: 16:39:24 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
[2026-04-02 16:39:24] INFO: 16:39:24 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
[2026-04-02 16:39:24] INFO: 16:39:24 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
[2026-04-02 16:39:24] INFO: 16:39:24 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-02 16:39:24] INFO: 16:39:24 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-02 16:39:24] INFO: 16:39:24 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
[20

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[2026-04-02 16:39:28] INFO: 16:39:28 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-02 16:39:28] INFO: 16:39:28 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-02 16:39:28] INFO: 16:39:28 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-02 16:39:28] INFO: 16:39:28 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-02 16:39:28] INFO: 16:39:28 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
[2026-04-02 16:39:28] INFO: 16:39:28 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/

In [23]:
# Step 3: Automated PDF Ingestion (Google Drive)
import os
import re
import requests

def extract_drive_file_id(drive_url: str) -> str:
    """Extract Google Drive file id from common share URL formats."""
    patterns = [
        r"/d/([A-Za-z0-9_-]+)",
        r"[?&]id=([A-Za-z0-9_-]+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, drive_url)
        if match:
            return match.group(1)
    raise ValueError("Could not extract a Google Drive file id from the provided link.")

def _raise_drive_access_error(file_id: str, status_code: int | None = None) -> None:
    status_msg = f"HTTP {status_code}. " if status_code else ""
    raise PermissionError(
        status_msg
        + "Google Drive blocked direct download for this file. "
        + "Set sharing to 'Anyone with the link: Viewer' and retry. "
        + f"You can also test with: https://drive.google.com/uc?export=download&id={file_id}"
    )

def download_pdf_from_drive(drive_url: str, save_path: str, session: requests.Session | None = None) -> None:
    """Download a Drive file and ensure the saved artifact is a valid PDF."""
    session = session or requests.Session()
    file_id = extract_drive_file_id(drive_url)

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0 Safari/537.36"
        ),
        "Accept": "text/html,application/pdf,application/octet-stream,*/*",
        "Referer": "https://drive.google.com/",
    }

    endpoints = [
        ("https://drive.google.com/uc", {"export": "download", "id": file_id}),
        ("https://drive.usercontent.google.com/download", {"id": file_id, "export": "download"}),
    ]

    last_status = None
    for base_url, params in endpoints:
        response = session.get(
            base_url,
            params=params,
            headers=headers,
            stream=True,
            allow_redirects=True,
            timeout=60,
        )
        last_status = response.status_code

        # Drive can require a confirm token for large files.
        token = None
        for key, val in response.cookies.items():
            if key.startswith("download_warning"):
                token = val
                break

        if not token and response.status_code < 400:
            preview_html = response.content[:200000].decode("utf-8", errors="ignore")
            token_match = re.search(r"confirm=([0-9A-Za-z-_]+)", preview_html)
            if token_match:
                token = token_match.group(1)

        if token:
            params = dict(params)
            params["confirm"] = token
            response = session.get(
                base_url,
                params=params,
                headers=headers,
                stream=True,
                allow_redirects=True,
                timeout=60,
            )
            last_status = response.status_code

        # If Drive still denies access, try next endpoint.
        if response.status_code in (401, 403):
            continue

        response.raise_for_status()

        with open(save_path, "wb") as file_obj:
            for chunk in response.iter_content(chunk_size=32768):
                if chunk:
                    file_obj.write(chunk)

        with open(save_path, "rb") as file_obj:
            header = file_obj.read(5)

        if header == b"%PDF-":
            console_log(f"Document secured: {save_path}", "OK")
            return

        with open(save_path, "rb") as file_obj:
            preview = file_obj.read(400).decode("utf-8", errors="ignore")
        os.remove(save_path)
        raise ValueError(
            "Downloaded file is not a valid PDF. "
            "Google Drive likely returned an HTML page (permissions/login/interstitial). "
            "Set sharing to 'Anyone with the link: Viewer' and retry. "
            f"Preview: {preview[:120]!r}"
        )

    _raise_drive_access_error(file_id=file_id, status_code=last_status)

def extract_text_or_ocr(pdf_path: str, dpi: int = 300) -> str:
    """Use PyMuPDF extraction first; fall back to OCR and return chosen source file path."""
    try:
        import fitz  # PyMuPDF
    except Exception:
        fitz = None
        console_log("PyMuPDF not available; OCR fallback may be required.", "WARN")

    extracted_text = ""
    if fitz is not None:
        try:
            with fitz.open(pdf_path) as doc:
                for page in doc:
                    extracted_text += page.get_text() + "\n"
        except Exception as exc:
            console_log(f"PyMuPDF extraction error: {exc}", "WARN")
            extracted_text = ""

    if len(extracted_text.strip()) > 50:
        console_log("PDF contains extractable text (PyMuPDF).", "OK")
        return pdf_path

    console_log("No reliable extractable text found; using OCR fallback.", "WARN")
    try:
        from pdf2image import convert_from_path
        import pytesseract
    except Exception:
        console_log(
            "Missing OCR packages/dependencies (pdf2image, pytesseract, Poppler, Tesseract).",
            "ERROR",
        )
        return pdf_path

    # Reuse notebook-level resolver so OCR works even when PATH is missing
    tess_cmd = resolve_tesseract_cmd() if "resolve_tesseract_cmd" in globals() else None
    if tess_cmd:
        console_log(f"Using Tesseract executable: {tess_cmd}", "INFO")

    try:
        _ = pytesseract.get_tesseract_version()
    except Exception:
        console_log(
            "Tesseract executable not found. Install it and rerun Step 3.",
            "ERROR",
        )
        return pdf_path

    try:
        images = convert_from_path(pdf_path, dpi=dpi)
        ocr_text = ""
        for image in images:
            ocr_text += pytesseract.image_to_string(image) + "\n"

        txt_path = pdf_path + ".ocr.txt"
        with open(txt_path, "w", encoding="utf-8") as file_obj:
            file_obj.write(ocr_text)

        console_log(f"OCR complete: {txt_path}", "OK")
        return txt_path
    except Exception as exc:
        console_log(f"OCR failed: {exc}", "ERROR")
        return pdf_path

drive_link = input("📌 Paste Google Drive Link: ").strip()
DATA_DIR = "academic_data"
os.makedirs(DATA_DIR, exist_ok=True)

pdf_path = os.path.join(DATA_DIR, "source_material.pdf")
download_pdf_from_drive(drive_link, pdf_path)

# source_file is consumed by Step 4 and can be either PDF or OCR text file
source_file = extract_text_or_ocr(pdf_path)
console_log(f"Using source file: {source_file}", "OK")

[2026-04-02 16:58:13] OK: Document secured: academic_data\source_material.pdf
[2026-04-02 16:58:14] OK: PDF contains extractable text (PyMuPDF).
[2026-04-02 16:58:14] OK: Using source file: academic_data\source_material.pdf


In [24]:
# Step 4: Semantic Chunking for Contextual Integrity
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SemanticSplitterNodeParser

# 'source_file' is produced by the previous cell and may be a PDF or a .ocr.txt file
documents = SimpleDirectoryReader(input_files=[source_file]).load_data()
parser = SemanticSplitterNodeParser(
    buffer_size=3,
    breakpoint_percentile_threshold=95,
    embed_model=embed_model
)

nodes = parser.get_nodes_from_documents(documents)

# Absolute Referencing Metadata
for n in nodes:
    n.metadata["source"] = os.path.basename(source_file)

console_log(f"Created {len(nodes)} high-quality semantic nodes from {os.path.basename(source_file)}.", "OK")


[2026-04-02 17:16:15] OK: Created 628 high-quality semantic nodes from source_material.pdf.


In [25]:
# Step 5: Advanced Query Fusion (Sequential Stability Mode)
import os
import sys
from pathlib import Path
import importlib.util

# 1. Clean up local path references
local_pack_root = Path("query_rewriting_pack").resolve()
if str(local_pack_root) not in sys.path:
    sys.path.insert(0, str(local_pack_root))

# 2. Robust Import Logic
try:
    from llama_index.packs.fusion_retriever.query_rewrite.base import QueryRewritingRetrieverPack
except Exception:
    # Fallback to direct file loading if the namespace is messy
    base_file = local_pack_root / "llama_index" / "packs" / "fusion_retriever" / "query_rewrite" / "base.py"
    spec = importlib.util.spec_from_file_location("local_query_rewrite_base", str(base_file))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    QueryRewritingRetrieverPack = module.QueryRewritingRetrieverPack

# 3. THE CRITICAL STABILITY FIX
# Setting num_queries=1 prevents the '8-way traffic jam' in your logs.
# This ensures a 1:1 ratio between your question and the API request.
query_rewriting_pack = QueryRewritingRetrieverPack(
    nodes,
    chunk_size=256,
    vector_similarity_top_k=5, 
    num_queries=1  # FORCE sequential processing
)

print("🚀 Search Engine set to SINGLE-STREAM mode.")
print("✅ Stability confirmed: Sequential processing enabled.")
console_log("Advanced Query Fusion Engine Ready (Sequential Fix)", "OK")

🚀 Search Engine set to SINGLE-STREAM mode.
✅ Stability confirmed: Sequential processing enabled.
[2026-04-02 17:17:33] OK: Advanced Query Fusion Engine Ready (Sequential Fix)


In [28]:
# Step 6: The Research Loop (Q&A) - Sequential Stability Version
import time

def _is_timeout_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    timeout_markers = [
        "read operation timed out",
        "timed out",
        "timeout",
        "readtimeout",
    ]
    return any(marker in msg for marker in timeout_markers)

def run_academic_query(question, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        try:
            # Since we set num_queries=1 in Step 5, this makes
            # a single, stable request to Granite 3.1 per attempt.
            response = query_rewriting_pack.run(question)

            # Convert to plain text for notebook-safe output handling.
            final_text = str(response).strip()
            if not final_text:
                return "⚠️ Information not found in source material."
            return final_text

        except Exception as e:
            error_msg = str(e).lower()
            if _is_timeout_error(e):
                if attempt < max_attempts:
                    wait_seconds = min(2 * attempt, 8)
                    print(
                        f"⚠️ Timeout detected (attempt {attempt}/{max_attempts}). "
                        f"Retrying in {wait_seconds}s..."
                    )
                    time.sleep(wait_seconds)
                    continue
                return (
                    "⚠️ CONNECTION TIMEOUT: The network/API stream timed out after "
                    f"{max_attempts} attempts. Please retry in a few seconds."
                )
            if "422" in error_msg:
                return "⚠️ MODEL ERROR: Granite 3.1 is currently busy. Wait 10 seconds and retry."
            return f"⚠️ SYSTEM ERROR: {e}"

print("\n" + "="*40)
print("🎓 ACADEMIC RESEARCH ENGINE: ONLINE")
print("Target: Absolute Grounding (80%+ Accuracy)")
print("Mode: Sequential Stability (Single-Stream)")
print("="*40)
print("Type 'end' to exit.")

while True:
    user_input = input("\n🟦 Research Question: ").strip()

    if not user_input:
        continue
    if user_input.lower() in ["end", "exit", "quit", "stop"]:
        print("🛑 Closing Research Engine. Good luck with your assignment!")
        break

    print("🔍 [1/1] Analyzing source_material.pdf... (This may take 15-30 seconds)")

    # We record the start time to monitor performance
    start_time = time.time()

    # Execute the query with retry protection for transient timeouts
    result = run_academic_query(user_input, max_attempts=3)

    duration = time.time() - start_time

    # Output results with clear academic formatting
    if "⚠️" in result:
        print(f"\n{result}")
    else:
        print(f"\n🧠 FACTUAL ANALYSIS (Granite 3.1):")
        print("-" * 30)
        print(result)
        print("-" * 30)
        print(f"⏱️ Analysis completed in {duration:.1f} seconds.")

    print("\n📍 REFERENCE: Absolute Reference to source_material.pdf")


🎓 ACADEMIC RESEARCH ENGINE: ONLINE
Target: Absolute Grounding (80%+ Accuracy)
Mode: Sequential Stability (Single-Stream)
Type 'end' to exit.
🔍 [1/1] Analyzing source_material.pdf... (This may take 15-30 seconds)
[2026-04-02 20:36:10] INFO: 20:36:10 - INFO - HTTP Request: POST https://api.replicate.com/v1/models/ibm-granite/granite-3.1-8b-instruct/predictions "HTTP/1.1 201 Created"
[2026-04-02 20:36:10] INFO: 20:36:10 - INFO - HTTP Request: POST https://api.replicate.com/v1/models/ibm-granite/granite-3.1-8b-instruct/predictions "HTTP/1.1 201 Created"
[2026-04-02 20:36:10] INFO: 20:36:10 - INFO - HTTP Request: POST https://api.replicate.com/v1/models/ibm-granite/granite-3.1-8b-instruct/predictions "HTTP/1.1 201 Created"
[2026-04-02 20:36:10] INFO: 20:36:10 - INFO - HTTP Request: POST https://api.replicate.com/v1/models/ibm-granite/granite-3.1-8b-instruct/predictions "HTTP/1.1 201 Created"
[2026-04-02 20:36:11] INFO: 20:36:11 - INFO - HTTP Request: GET https://stream-b.svc.ric2.c.replica